# U1-01 · Python, NumPy, pandas y Matplotlib con datos de la economía

*Machine Learning aplicado usando Python (IFCD093PO) · Unidad 1, apartados 1.1 a 1.3*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador, sin instalar nada. El archivo `macrodata.csv` está junto a este cuaderno.

> Datos: *macrodata*, incluidos en statsmodels; Reserva Federal de San Luis (FRED) y Oficina de Estadísticas Laborales de EE. UU. Dominio público.

## 1. Python sin bibliotecas

In [ ]:
import csv

with open("macrodata.csv") as f:          # 203 trimestres, 1959-2009
    filas = list(csv.DictReader(f))
print(len(filas), "trimestres; el primero:", filas[0]["year"],
      filas[0]["quarter"], "PIB", filas[0]["realgdp"])

def crecimiento(antes, despues):
    """Variación en porcentaje entre dos valores."""
    return 100 * (despues - antes) / antes

pib = [float(f["realgdp"]) for f in filas]     # comprensión de lista
crec = [crecimiento(a, b) for a, b in zip(pib, pib[1:])]
print(f"crecimiento trimestral medio: {sum(crec) / len(crec):.2f} %")

# recesión técnica: dos trimestres seguidos con el PIB a la baja
recesiones = []
for i in range(1, len(crec)):
    if crec[i - 1] < 0 and crec[i] < 0:
        f = filas[i + 1]                       # crec[i] acaba en i+1
        recesiones.append(f"{f['year']}T{f['quarter']}")
print("trimestres en recesión técnica:", recesiones)

paro = {}                                      # década -> valores
for f in filas:
    decada = int(f["year"]) // 10 * 10
    paro.setdefault(decada, []).append(float(f["unemp"]))
for decada, v in paro.items():
    print(f"años {decada}: paro medio {sum(v) / len(v):.1f} %")

**Prueba.** Cambia la definición de recesión a *tres* trimestres seguidos con el PIB a la baja. ¿Cuántos episodios quedan?

## 2. NumPy: operar con vectores

En el navegador los tiempos son distintos de los de un ordenador con Python instalado.

In [ ]:
import time
import numpy as np

pib = np.loadtxt("macrodata.csv", delimiter=",", skiprows=1,
                 usecols=2)                    # columna realgdp
crec = 100 * (pib[1:] - pib[:-1]) / pib[:-1]   # todo a la vez
print(pib.shape, f"media {crec.mean():.2f} %, mínimo "
      f"{crec.min():.2f} %, máximo {crec.max():.2f} %")
recesion = (crec[:-1] < 0) & (crec[1:] < 0)    # dos caídas seguidas
print("trimestres en recesión técnica:", recesion.sum())

# la misma cuenta repetida 10 000 veces: bucle frente a NumPy
t0 = time.perf_counter()
for _ in range(10_000):
    c = [100 * (b - a) / a for a, b in zip(pib, pib[1:])]
t1 = time.perf_counter()
for _ in range(10_000):
    c = 100 * (pib[1:] - pib[:-1]) / pib[:-1]
t2 = time.perf_counter()
print(f"bucle {t1 - t0:.2f} s, NumPy {t2 - t1:.3f} s: "
      f"{(t1 - t0) / (t2 - t1):.0f} veces más rápido")

## 3. pandas y Matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

d = pd.read_csv("macrodata.csv")
d.index = pd.PeriodIndex.from_fields(year=d["year"],
                                     quarter=d["quarter"], freq="Q")
# crecimiento anual: cada trimestre frente al mismo de un año antes
d["crec_anual"] = 100 * d["realgdp"].pct_change(4)
print(d[["realgdp", "crec_anual", "unemp", "infl"]].tail(3))
decada = d["year"] // 10 * 10
decadas = d.groupby(decada)[["unemp", "infl"]].mean()
print(decadas.round(1))
peor = d["crec_anual"].idxmin()
print(f"peor año: el que acaba en {peor}, PIB "
      f"{d.loc[peor, 'crec_anual']:.1f} %")

fig, eje = plt.subplots(figsize=(4.4, 2.0))
eje.plot(d.index.to_timestamp(), d["unemp"], label="paro (%)")
eje.plot(d.index.to_timestamp(), d["infl"], label="inflación (%)",
         lw=0.8)
for t in d.index[d["crec_anual"] < 0]:        # años de PIB a la baja
    eje.axvspan(t.start_time, t.end_time, color="0.88", lw=0)
eje.legend(); plt.show()  # fig.savefig("macro.pdf")

**Prueba.** Calcula el paro máximo de cada década con `.max()` en lugar de `.mean()`.

## 4. Primer caso: el paro y el crecimiento

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error as eam

d = pd.read_csv("macrodata.csv")
d["g"] = 100 * d["realgdp"].pct_change(4)   # crecimiento anual (%)
d["du"] = d["unemp"].diff(4)          # cambio del paro, en puntos
d = d.dropna()
ent = d[d["year"] < 1990]                    # el pasado: aprende
pru = d[d["year"] >= 1990]                   # el futuro: prueba
X, y, Xp, yp = ent[["g"]], ent["du"], pru[["g"]], pru["du"]
print(f"entrenamiento {len(ent)} trimestres, prueba {len(pru)}")

def regla(g):                # escrita a mano, con números redondos
    return -0.5 * (g - 3)

modelo = LinearRegression().fit(X, y)        # aprende de los datos
b, a = modelo.coef_[0], modelo.intercept_
print(f"aprendido: cambio del paro = {b:.3f} * g {a:+.3f}")
print(f"el paro no cambia si el PIB crece {-a / b:.2f} %")

for nombre, pred in [("referencia: sin cambios", 0 * yp),
                     ("regla a mano", regla(pru["g"])),
                     ("modelo aprendido", modelo.predict(Xp))]:
    print(f"{nombre:<24} error medio {eam(yp, pred):.3f} puntos")

g = np.linspace(-5, 9, 2)
plt.scatter(ent["g"], ent["du"], s=4, label="1960-1989")
plt.scatter(pru["g"], pru["du"], s=4, label="1990-2009")
plt.plot(g, b * g + a, "k", lw=1, label="aprendido")
plt.plot(g, regla(g), "k--", lw=1, label="regla a mano")
plt.xlabel("crecimiento del PIB en un año (%)")
plt.ylabel("cambio del paro (puntos)")
plt.legend(); plt.show()  # plt.savefig("okun.pdf")

**Ejercicio.** Entrena el modelo con los años 1960-1979 y prueba con 1980-2009. ¿Sigue ganando la regla escrita a mano?

> Ball, L., Leigh, D. y Loungani, P. (2017). Okun's Law: Fit at 50? *Journal of Money, Credit and Banking*, 49(7), 1413–1441. https://doi.org/10.1111/jmcb.12420